# 02. 은닉층과 옵티마이저
같은 데이터를 사용하여 모델 구성 방법, Flatten, 활성화함수, 학습률을 살펴봅니다.
이 노트북은 01의 실행 상태에 의존하지 않습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from common.runtime import configure_runtime
from common.data import load_fashion_data

gpus = configure_runtime(seed=42)
print('GPU:', gpus)

In [ ]:
(train_x, train_y), (val_x, val_y), (test_x, test_y) = load_fashion_data()
print('훈련:', train_x.shape, train_y.shape)
print('검증:', val_x.shape, val_y.shape)
print('테스트:', test_x.shape, test_y.shape)

## Sequential을 구성하는 세 가지 방법
각 모델은 `784 → 100 → 10` 구조입니다. Dense(100)은 단순히 중요한 100개 원본 특성을 고르는 것이 아니라
학습된 가중치로 새로운 표현을 만듭니다. 세 예제는 구조만 확인하고 마지막 예제를 학습합니다.

In [ ]:
# 1. 계층을 먼저 만들어 전달하기
hidden = keras.layers.Dense(100, activation='sigmoid')
output = keras.layers.Dense(10, activation='softmax')
model_layers = keras.Sequential([keras.Input(shape=(784,)), hidden, output])

# 2. 생성자에 계층 목록을 직접 전달하기
model_list = keras.Sequential([
    keras.Input(shape=(784,)),
    keras.layers.Dense(100, activation='sigmoid'),
    keras.layers.Dense(10, activation='softmax'),
])

# 3. add()로 순서대로 추가하기
model_add = keras.Sequential()
model_add.add(keras.Input(shape=(784,)))
model_add.add(keras.layers.Dense(100, activation='sigmoid'))
model_add.add(keras.layers.Dense(10, activation='softmax'))
model_add.summary()

In [ ]:
EPOCHS = 10  # 원본의 50~200회 반복 학습을 짧은 독립 실험으로 정리
model_add.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
sigmoid_history = model_add.fit(
    train_x.reshape(-1, 784), train_y, epochs=EPOCHS,
    validation_data=(val_x.reshape(-1, 784), val_y), verbose=2)
print('sigmoid 검증:', model_add.evaluate(val_x.reshape(-1, 784), val_y, verbose=0))

## Flatten과 ReLU
Flatten은 `(28, 28)`을 `(784,)`로 펼치는 계층입니다. 학습 파라미터는 없지만
전처리 선택과 입력 표현은 모델의 동작과 성능에 영향을 줄 수 있습니다.
ReLU는 음수를 0으로, 양수는 그대로 출력하는 비선형 함수입니다.
원본처럼 이미 학습한 모델을 재컴파일해 비교하면 이전 가중치가 남으므로, 각 비교에서 새 모델을 만듭니다.

In [ ]:
def make_model():
    return keras.Sequential([
        keras.Input(shape=(28, 28)),
        keras.layers.Flatten(),
        keras.layers.Dense(100, activation='relu'),
        keras.layers.Dense(10, activation='softmax'),
    ])

relu_model = make_model()
relu_model.summary()
relu_model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
relu_history = relu_model.fit(train_x, train_y, epochs=EPOCHS,
    validation_data=(val_x, val_y), verbose=2)

## 옵티마이저와 학습률
| 방법 | 핵심 |
| --- | --- |
| SGD | 미니배치 기울기를 사용해 갱신 |
| Momentum | SGD에 이전 갱신 방향을 반영 |
| Adagrad | 누적된 기울기 제곱으로 각 파라미터의 보폭을 조절 |
| RMSprop | 기울기 제곱의 이동평균을 사용 |
| Adam | 기울기와 기울기 제곱의 이동평균을 사용 |

방법 사이에 고정된 성능 순위는 없습니다. 학습률 기본값도 옵티마이저마다 다릅니다.
학습률을 높이면 과대적합이, 낮추면 과소적합이 해결된다는 규칙은 없습니다.
검증 성능으로 비교하고 테스트 데이터는 보지 않습니다.

In [ ]:
# 원본의 4개 옵티마이저 × 학습률 2개 비교 과제.
# 기본은 짧은 비교이며, 수업 설정은 COMPARE_EPOCHS=50으로 재현합니다.
COMPARE_EPOCHS = 3
optimizer_types = {
    'SGD': keras.optimizers.SGD, 'Adagrad': keras.optimizers.Adagrad,
    'RMSprop': keras.optimizers.RMSprop, 'Adam': keras.optimizers.Adam,
}
comparison = []
for name, optimizer_type in optimizer_types.items():
    for learning_rate in (0.1, 0.01):
        keras.backend.clear_session()
        keras.utils.set_random_seed(42)
        candidate = make_model()
        candidate.compile(optimizer=optimizer_type(learning_rate=learning_rate),
            loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        candidate.fit(train_x, train_y, epochs=COMPARE_EPOCHS,
            validation_data=(val_x, val_y), verbose=0)
        loss, accuracy = candidate.evaluate(val_x, val_y, verbose=0)
        comparison.append((name, learning_rate, loss, accuracy))
for name, lr, loss, accuracy in comparison:
    print(f'{name:8s} lr={lr:<5} val_loss={loss:.4f} val_accuracy={accuracy:.4f}')

## 복습
조건을 맞추기 위해 같은 분할·초기 난수·학습 횟수로 비교합니다.
짧은 학습만으로 우열을 단정하지 않고 학습곡선과 학습률 조합을 확인합니다.